## Task 3 - Simulación Numérica
**a. Implementación del sistema en Python**
Se implementa el sistema de Ecuaciones Diferenciales Ordinarias (EDOs) haciendo uso de `NumPy`, también se implemento manualmente los métodos de integración numérica de Euler y Runge-Kutta de cuarto orden (RK4).

In [1]:
import numpy as np

# Diapositivas 1-2: contexto general de modelación y motivación del uso de modelos dinámicos.
# Diapositivas 3-4: traducción del diagrama de stocks/flows a ecuaciones diferenciales.
# Diapositivas 5-7: estructura de sistema dinámico con stocks acoplados y validación conceptual.
# Parámetros del modelo 
N = 10000.0
beta = 0.02
gamma = 0.3
delta = 0.05
epsilon = 0.1
T = 100.0 # Semanas

# Condiciones iniciales: [S(0), A(0), R(0)]
y0 = np.array([9800.0, 200.0, 0.0])

def derivadas(t, y):
    """
    Función que calcula las derivadas (flows netos) para cada stock.
    y es un arreglo [S, A, R]
    """
    S, A, R = y
    
    # Diapositivas 3-4: cada flow entra/sale de un stock y forma la EDO correspondiente.
    # Ecuaciones de los flows
    dS_dt = -beta * S - gamma * S * A / N
    dA_dt = beta * S + gamma * S * A / N + epsilon * R * A / N - delta * A
    dR_dt = delta * A - epsilon * R * A / N
    
    return np.array([dS_dt, dA_dt, dR_dt])

# Diapositiva 8: resolver EDOs de forma numérica con integración paso a paso.
# Diapositiva 9: implementación de Euler (método de primer orden).
def euler(f, y0, t0, tf, dt):
    """Implementación manual del método de Euler"""
    t = np.arange(t0, tf + dt, dt)
    y = np.zeros((len(t), len(y0)))
    y[0] = y0
    for i in range(len(t) - 1):
        y[i+1] = y[i] + f(t[i], y[i]) * dt
    return t, y

# Diapositivas 10-11 y 14: RK4 para mayor precisión y comparación frente a Euler.
def rk4(f, y0, t0, tf, dt):
    """Implementación manual del método de Runge-Kutta 4 (RK4)"""
    t = np.arange(t0, tf + dt, dt)
    y = np.zeros((len(t), len(y0)))
    y[0] = y0
    for i in range(len(t) - 1):
        k1 = f(t[i], y[i])
        k2 = f(t[i] + dt/2.0, y[i] + (dt/2.0) * k1)
        k3 = f(t[i] + dt/2.0, y[i] + (dt/2.0) * k2)
        k4 = f(t[i] + dt, y[i] + dt * k3)
        y[i+1] = y[i] + (dt/6.0) * (k1 + 2*k2 + 2*k3 + k4)
    return t, y

**b. y c. Corridas de Simulación y Comparación de Métodos**
Se hace el modelo que se uso en el método de Euler con pasos $\Delta t \in \{1.0, 0.5, 0.1\}$ y el método RK4 con $\Delta t = 1.0$

In [2]:
# Diapositivas 10-11 y 14: corridas para comparar precisión y comportamiento entre métodos.
# Diapositiva 13: análisis del efecto del tamaño de paso (dt) sobre el error numérico.
# b. Corridas con método de Euler
t_e1, y_e1 = euler(derivadas, y0, 0, T, 1.0)
t_e05, y_e05 = euler(derivadas, y0, 0, T, 0.5)
t_e01, y_e01 = euler(derivadas, y0, 0, T, 0.1)

# Extraer el valor final de A(T) para cada corrida (índice 1 de y)
A_final_e1 = y_e1[-1, 1]
A_final_e05 = y_e05[-1, 1]
A_final_e01 = y_e01[-1, 1]

print(f"Euler (dt=1.0) - A(T) final: {A_final_e1:.4f}")
print(f"Euler (dt=0.5) - A(T) final: {A_final_e05:.4f}")
print(f"Euler (dt=0.1) - A(T) final: {A_final_e01:.4f}")

# Diapositivas 10-11 y 14: corrida de referencia con RK4 para comparación.
# c. Corrida con método RK4
t_rk4, y_rk4 = rk4(derivadas, y0, 0, T, 1.0)
A_final_rk4 = y_rk4[-1, 1]

print(f"\nRK4 (dt=1.0) - A(T) final: {A_final_rk4:.4f}")

Euler (dt=1.0) - A(T) final: 5024.9990
Euler (dt=0.5) - A(T) final: 5026.4586
Euler (dt=0.1) - A(T) final: 5027.6453

RK4 (dt=1.0) - A(T) final: 5027.9447


**Respuesta a inciso c:**

- *Comparación:* El resultado de `A(T)` bajo el método RK4 con $\Delta t = 1.0$ coincide de forma mucho más cercana con la corrida de Euler de $\Delta t = 0.1$

- *Conclusión de eficiencia:* Esto demuestra que RK4 es mucho más eficiente que Euler.  Euler tiene un error global proporcional a $\Delta t$ y requiere pasos muy pequeños para ser preciso, mientras que RK4, al ser de cuarto orden, logra una altísima precisión utilizando pasos mucho más grandes, lo que ahorra esfuerzo computacional.

**d. Verificación de la Conservación de la Población**

In [3]:
# Diapositiva 7: verificación de consistencia del modelo (conservación del stock total).
# Diapositiva 13: cuantificación del error numérico acumulado por redondeo.
# Se comprueba la conservación en la corrida de RK4 (dt=1.0)
# Sumamos S + A + R en cada paso de tiempo
poblacion_total = np.sum(y_rk4, axis=1)

# Verificamos si alguna vez la diferencia con N es mayor a la tolerancia de la máquina
violaciones = np.where(np.abs(poblacion_total - N) > 1e-10)[0]

if len(violaciones) > 0:
    print(f"La conservación se violó en {len(violaciones)} pasos.")
else:
    print("La conservación se mantiene matemáticamente (dentro del límite de precisión de coma flotante).")
    
# Ver el error máximo de redondeo
error_max = np.max(np.abs(poblacion_total - N))
print(f"Error numérico máximo respecto a N: {error_max}")

La conservación se mantiene matemáticamente (dentro del límite de precisión de coma flotante).
Error numérico máximo respecto a N: 3.637978807091713e-12


**Respuesta a inciso d:**
Dado que la suma de nuestras ecuaciones diferenciales es algebraicamente exacta a cero ($\frac{dS}{dt} + \frac{dA}{dt} + \frac{dR}{dt} = 0$), métodos lineales iterativos como Euler y RK4 preservan invariantes lineales teóricamente de forma perfecta. 

Pero al implementarlos en una computadora, la condición estricta $S(t) + A(t) + R(t) = 10000$ se viola infinitesimalmente debido al **error de redondeo de punto flotante** (machine precision) inherente al hardward. Estos errores son del orden de $10^{-14}$. 

- *Implicaciones:* Aunque esta violación es despreciable para la escala, en sistemas con miles de millones de iteraciones o donde pequeñas desviaciones se amplifiquen por retroalimentación positiva, el error numérico acumulado puede llevar a conclusiones de comportamiento divergentes e irreales

**e. Recomendación de Política (Resumen Ejecutivo)**

Nuestro análisis del modelo demuestra que duplicar la intensidad de la campaña institucional inicial **no** aumentará la cantidad máxima de personas que adoptan el protocolo a largo plazo, únicamente acelerará la velocidad con la que se alcanza el estancamiento. La adopción sostenida depende estrictamente del balance entre el abandono por fatiga y la reactivación comunitaria. Bajo las métricas actuales, se estabilizará permanentemente en el 50% de la población (5,000 personas). Para maximizar y sostener la adopción, sugiero reasignar el presupuesto institucional. En lugar de financiar campañas de captación inicial, recomiendo invertir en estrategias comunitarias focalizadas que fomenten el refuerzo social entre adoptantes y ex-adoptantes, e implementar medidas que reduzcan la fatiga. Fortalecer el ecosistema de contacto directo (reactivación) es matemáticamente la única vía para elevar el techo estructural del sistema.